In [ ]:
from langchain_openai import ChatOpenAI
from langchain_unstructured import UnstructuredLoader
from langchain.text_splitter import CharacterTextSplitter
from langchain.vectorstores import FAISS
from langchain.vectorstores.utils import filter_complex_metadata
from langchain_openai import OpenAIEmbeddings
from langchain.embeddings import CacheBackedEmbeddings
from langchain.storage import LocalFileStore
from langchain.prompts import ChatPromptTemplate
from langchain.schema.runnable import RunnablePassthrough, RunnableLambda
from langchain.document_loaders import UnstructuredFileLoader


# llm
llm = ChatOpenAI(temperature=0.1)

# document
splitter = CharacterTextSplitter.from_tiktoken_encoder(
    chunk_size=600, chunk_overlap=100, separator="\n"
)
loader = UnstructuredLoader(
    file_path="./files/chapter_one.docx",
)


docs = loader.load_and_split()

print(len(docs))


# embedding
cache_store = LocalFileStore("./.cache/")
embeddings = OpenAIEmbeddings()
cached_embeddings = CacheBackedEmbeddings.from_bytes_store(embeddings, cache_store)


# vector store
vector_store = FAISS.from_documents(docs, cached_embeddings)


# retriever
retriever = vector_store.as_retriever()


# prompt
map_doc_template = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """
            Use the following portion of a long document to see if any of the text is relevant to answer the question. Return any relevant text verbatim.
            ------
            {context}
            """,
        ),
        ("human", "{question}"),
    ]
)

map_doc_chain = map_doc_template | llm


def map_docs(inputs):
    documents = inputs["document"]
    question = inputs["question"]
    results = []
    for document in documents:
        result = map_doc_chain.invoke(
            {"context": document.page_content, "question": question}
        ).content
        results.append(result)
    results = "\n\n".join(results)
    return results

    # return "\n\n".join(
    #     map_doc_chain.invoke(
    #         {"context": doc.page_content, "question": question}
    #     ).content
    #     for doc in documents
    # )


map_chain = {"document": retriever, "question": RunnablePassthrough()} | RunnableLambda(
    map_docs
)


final_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """
            Given the following extracted parts of a long document and a question, create a final answer.
            If you don't know the answer, just say you don't know. Don't try to make up an answer
            ------
            {context}
            """,
        ),
        ("human", "{question}"),
    ]
)

chain = {"context": map_chain, "question": RunnablePassthrough()} | final_prompt | llm


chain.invoke("Describe Victory Mansions.")

59
